In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# ── Load datasets ────────────────────────────────────────────────────────────
print("Loading datasets...")
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")

Loading datasets...
Train shape : (77299, 11)
Test shape  : (41778, 10)


In [2]:
def preprocess_base(df, is_train=True):
    df = df.copy()

    # ── 1. Temporal features ──────────────────────────────────────────────────
    df['Hour']   = df['timestamp'].apply(lambda x: int(str(x).split(':')[0]))
    df['Minute'] = df['timestamp'].apply(lambda x: int(str(x).split(':')[1]))

    # Cyclical encoding for Hour (smooth 24-h periodicity)
    df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24.0)
    df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24.0)

    # Cyclical encoding for full time-of-day in minutes (0-1425, 15-min grid)
    # Captures finer intra-hour demand rhythms that raw Minute misses
    tod = df['Hour'] * 60 + df['Minute']
    df['Time_sin'] = np.sin(2 * np.pi * tod / (24 * 60))
    df['Time_cos'] = np.cos(2 * np.pi * tod / (24 * 60))

    # ── 2. Day feature ────────────────────────────────────────────────────────
    # Only 2 days in train (48, 49); test has only day 49.
    # Keep as-is — XGBoost will pick up the signal if it exists.
    # (No dow mod needed with 2 unique values.)

    # ── 3. Fill missing values ────────────────────────────────────────────────
    df['RoadType']    = df['RoadType'].fillna('Unknown')
    df['Weather']     = df['Weather'].fillna('Unknown')
    df['Temperature'] = df['Temperature'].fillna(
        df['Temperature'].median() if is_train else 25.0
    )

    # ── 4. Label encode categoricals ─────────────────────────────────────────
    for col in ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks']:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    # ── 5. Drop raw/index columns (keep geohash for target encoding later) ────
    df = df.drop(columns=['Index', 'timestamp'])

    return df


print("Running base preprocessing...")
processed_train = preprocess_base(train_df, is_train=True)
processed_test  = preprocess_base(test_df,  is_train=False)

X          = processed_train.drop(columns=['demand'])
y          = processed_train['demand']
X_test_base = processed_test.copy()

print("Preprocessing complete.")
print(f"Feature columns ({len(X.columns)}): {list(X.columns)}")

Running base preprocessing...
Preprocessing complete.
Feature columns (14): ['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'Hour', 'Minute', 'Hour_sin', 'Hour_cos', 'Time_sin', 'Time_cos']


In [3]:
print("Splitting data and applying leak-free Target Encoding...")

# ── 1. Strict train/val split FIRST (no leakage from full dataset) ───────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Log-transform targets to handle heavy right-skew in demand
y_train_log = np.log1p(y_train)
y_val_log   = np.log1p(y_val)
overall_mean = y_train_log.mean()

# ── 2. Build encoding maps from TRAIN fold only ───────────────────────────────

train_enc = X_train.copy()
train_enc['target_log'] = y_train_log.values

# (a) Geohash mean demand  → primary location signal
geo_mean = train_enc.groupby('geohash')['target_log'].mean()

# (b) Geohash × Hour mean  → captures time-varying location demand
#     This is the single biggest improvement over the baseline.
#     Coverage: ~70 % of geo-hour pairs exist; rest fall back to geo_mean.
train_enc['geo_hour_key'] = (
    train_enc['geohash'] + '_' + train_enc['Hour'].astype(str)
)
geo_hour_mean = train_enc.groupby('geo_hour_key')['target_log'].mean()

# (c) Geohash prefix-4 mean → fallback for the ~10 OOV geohashes in test
train_enc['geo_prefix4'] = train_enc['geohash'].str[:4]
geo_prefix4_mean = train_enc.groupby('geo_prefix4')['target_log'].mean()


# ── 3. Apply encodings with layered fallback: geo_hour → geo → prefix4 → mean ─

def apply_encodings(df, geo_mean, geo_hour_mean, geo_prefix4_mean, overall_mean):
    df = df.copy()

    # Primary: per-geohash mean
    df['Geo_Mean'] = df['geohash'].map(geo_mean).fillna(
        df['geohash'].str[:4].map(geo_prefix4_mean)
    ).fillna(overall_mean)

    # Primary interaction: geohash × hour
    gh_key = df['geohash'] + '_' + df['Hour'].astype(str)
    df['Geo_Hour_Mean'] = gh_key.map(geo_hour_mean).fillna(
        df['Geo_Mean']   # graceful fallback to geo mean for unseen combos
    )

    df = df.drop(columns=['geohash'])
    return df


X_train = apply_encodings(X_train, geo_mean, geo_hour_mean, geo_prefix4_mean, overall_mean)
X_val   = apply_encodings(X_val,   geo_mean, geo_hour_mean, geo_prefix4_mean, overall_mean)
X_test_final = apply_encodings(X_test_base, geo_mean, geo_hour_mean, geo_prefix4_mean, overall_mean)

print(f"Final feature set ({len(X_train.columns)}): {list(X_train.columns)}")
print("\nTraining XGBoost...")

# ── 4. XGBoost — tuned with regularization to prevent overfitting ─────────────
#   max_depth reduced 9→7 to compensate for the richer feature set
#   reg_alpha + reg_lambda penalise the many encoding features
model = xgb.XGBRegressor(
    n_estimators      = 2000,
    max_depth         = 7,
    learning_rate     = 0.03,
    subsample         = 0.85,
    colsample_bytree  = 0.85,
    min_child_weight  = 3,
    reg_alpha         = 0.1,   # L1 — sparse feature selection
    reg_lambda        = 1.5,   # L2 — smooth weight shrinkage
    random_state      = 42,
    tree_method       = 'hist',
    early_stopping_rounds = 100
)

model.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    verbose=200
)

# ── 5. Evaluate ───────────────────────────────────────────────────────────────
val_preds = np.expm1(model.predict(X_val))
y_val_act = np.expm1(y_val_log)

r2   = r2_score(y_val_act, np.clip(val_preds, 0, None))
rmse = np.sqrt(mean_squared_error(y_val_act, np.clip(val_preds, 0, None)))

print("\n=== Validation Performance ===")
print(f"R2 Score : {r2 * 100:.2f}%")
print(f"RMSE     : {rmse:.5f}")

Splitting data and applying leak-free Target Encoding...
Final feature set (15): ['day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'Hour', 'Minute', 'Hour_sin', 'Hour_cos', 'Time_sin', 'Time_cos', 'Geo_Mean', 'Geo_Hour_Mean']

Training XGBoost...
[0]	validation_0-rmse:0.10620
[200]	validation_0-rmse:0.02581
[301]	validation_0-rmse:0.02584

=== Validation Performance ===
R2 Score : 95.13%
RMSE     : 0.03139


In [4]:
print("Generating final test predictions...")

# Predict and reverse log transform
test_preds_log = model.predict(X_test_final)
test_preds = np.expm1(test_preds_log)
test_preds = np.clip(test_preds, 0, None)   # No negative traffic demand

# Build submission
submission = pd.DataFrame({
    'Index' : test_df['Index'],
    'demand': test_preds
})

submission.to_csv('final_submission.csv', index=False)

print(f"Rows submitted : {len(submission)}")
print(f"Demand range   : {test_preds.min():.5f} – {test_preds.max():.5f}")
print("Submission file 'final_submission.csv' saved successfully!")

Generating final test predictions...
Rows submitted : 41778
Demand range   : 0.00161 – 1.01201
Submission file 'final_submission.csv' saved successfully!
